# create_dataset.ipynb
Downloads S&P 500 OHLCV data from Yahoo Finance, cleans it, computes log returns,
and saves everything to disk for use in the features notebook.

In [31]:
import requests
import os
from io import StringIO
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [32]:
headers = {"User-Agent": "Mozilla/5.0"}
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

response = requests.get(url, headers=headers)
tables = pd.read_html(StringIO(response.text))
sp500 = tables[0]
tickers = sp500["Symbol"].tolist()
tickers = [t.replace(".", "-") for t in tickers]

sector_map = {}
if "GICS Sector" in sp500.columns:
    sector_map = dict(zip(sp500["Symbol"].str.replace(".", "-"), sp500["GICS Sector"]))

print(f"found {len(tickers)} tickers")
print(tickers[:5])

found 503 tickers
['MMM', 'AOS', 'ABT', 'ABBV', 'ACN']


In [33]:
# ── Download full OHLCV data ───────────────────────────────────────────────────
# We cache so repeated runs are instant.
# We now download ALL price fields (Open, High, Low, Close, Volume) so the
# features notebook can build volume-based signals and True Range.

os.makedirs("data", exist_ok=True)

RAW_PATH = "data/ohlcv_raw.parquet"

if os.path.exists(RAW_PATH):
    raw = pd.read_parquet(RAW_PATH)
    print(f"loaded from disk: {raw.shape}")
else:
    # yfinance returns a MultiIndex DataFrame when multiple tickers are requested
    raw = yf.download(
        tickers,
        start="2015-01-01",
        end="2025-12-31",
        auto_adjust=True,   # adjusts OHLC & Volume for splits/dividends
        group_by="ticker",  # shape: (dates, (ticker, field))
    )
    raw.to_parquet(RAW_PATH)
    print(f"downloaded and saved: {raw.shape}")

loaded from disk: (2765, 2515)


In [34]:
# ── Reshape to a dict of DataFrames, one per price field ──────────────────────
# raw columns look like  ('AAPL', 'Close'), ('AAPL', 'Open'), ...

fields = ["Open", "High", "Low", "Close", "Volume"]
price_data = {}

for field in fields:
    try:
        df = raw.xs(field, axis=1, level=1)   # shape: (dates, tickers)
    except KeyError:
        # Fallback for some yfinance versions that swap level order
        df = raw.xs(field, axis=1, level=0)
    price_data[field] = df

close = price_data["Close"]
print(f"Close shape: {close.shape}")
print(f"Fields available: {list(price_data.keys())}")

Close shape: (2765, 503)
Fields available: ['Open', 'High', 'Low', 'Close', 'Volume']


In [35]:
# ── Drop tickers with > 20 % missing Close prices ─────────────────────────────
min_count = int(0.8 * len(close))
valid_tickers = close.columns[close.notna().sum() >= min_count].tolist()

print(f"tickers before: {close.shape[1]}")
print(f"tickers after:  {len(valid_tickers)}")

# Keep only valid tickers across all fields
price_data_clean = {}
for field in fields:
    df = price_data[field][valid_tickers]
    df = df.ffill()        # fill small gaps
    df = df.dropna(axis=1) # drop any remaining all-NaN columns
    price_data_clean[field] = df

# Align all fields to the same ticker universe (intersection)
common_tickers = list(
    set.intersection(*[set(price_data_clean[f].columns) for f in fields])
)
for field in fields:
    price_data_clean[field] = price_data_clean[field][common_tickers]

close_clean  = price_data_clean["Close"]
open_clean   = price_data_clean["Open"]
high_clean   = price_data_clean["High"]
low_clean    = price_data_clean["Low"]
volume_clean = price_data_clean["Volume"]

print(f"Final ticker universe: {len(common_tickers)}")
print(f"Any NaNs in Close: {close_clean.isnull().any().any()}")

tickers before: 503
tickers after:  474
Final ticker universe: 462
Any NaNs in Close: False


In [36]:
# ── Compute log returns from adjusted close ────────────────────────────────────
log_returns = np.log(close_clean / close_clean.shift(1))
log_returns = log_returns.dropna(how="all")

print(f"log_returns shape: {log_returns.shape}")
print(log_returns.head())

log_returns shape: (2764, 462)
Ticker           CRL       AES       JNJ      FSLR         T       FCX  \
Date                                                                     
2015-01-05  0.007797 -0.028880 -0.007008 -0.062998 -0.009493 -0.061289   
2015-01-06  0.004185 -0.022797 -0.004926 -0.023462  0.001490  0.017010   
2015-01-07  0.023692  0.001536  0.021836  0.021548  0.005813  0.014541   
2015-01-08  0.005423  0.015233  0.007832  0.044046  0.009900  0.022064   
2015-01-09  0.002551 -0.025260 -0.013723  0.012980 -0.002990  0.003417   

Ticker           PCG       NUE      DECK       STT  ...       GIS       NOC  \
Date                                                ...                       
2015-01-05  0.016732 -0.038903 -0.007606 -0.016732  ... -0.018241 -0.021324   
2015-01-06 -0.001476 -0.014530 -0.017557 -0.028968  ... -0.002496  0.005495   
2015-01-07  0.007908  0.009213  0.031938  0.009309  ...  0.020925  0.031142   
2015-01-08  0.012561  0.024022  0.039710  0.020956  ...

In [37]:
# ── Save everything to disk ───────────────────────────────────────────────────
os.makedirs("data/processed", exist_ok=True)

log_returns.to_csv("data/log_returns.csv")

# Save individual price series as CSVs (aligned to log_returns index where applicable)
close_clean.to_csv("data/close.csv")
open_clean.to_csv("data/open.csv")
high_clean.to_csv("data/high.csv")
low_clean.to_csv("data/low.csv")
volume_clean.to_csv("data/volume.csv")

# Save sector mapping for the valid ticker universe
sector_series = pd.Series(
    {t: sector_map.get(t, "Unknown") for t in common_tickers},
    name="sector"
)
sector_series.to_csv("data/sector_map.csv")

print("saved: log_returns, close, open, high, low, volume, sector_map")

saved: log_returns, close, open, high, low, volume, sector_map
